<a href="https://colab.research.google.com/github/Lookieman/MSAI/blob/main/AI6130/AI6130_Grp/Iter1_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!pip install langchain langchain-community sentence-transformers faiss-cpu torch

In [ ]:
import os
import logging
import torch
import faiss
import numpy as np
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional

# For document loading and processing
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# For embeddings
from sentence_transformers import SentenceTransformer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/AI6130_Grp

In [ ]:
# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler("rag_system.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

In [ ]:
class RAGSystem:

  def __init__(self, embedding_model_name: str = "BAAI/bge-small-en-v1.5", chunz_size: int = 1500, chunk_overlap: int = 300):

    #init param
    self.embedding_model_name = embedding_model_name
    self.chunk_size = chunk_size
    self.chunk_overlap = chunk_overlap

    #init storage for document chunks and metadata

    self.documents = []
    self.meatadatas = []
    self.embeddings = None
    self.embedding_model = None
    self.index = None

    #load embedding model
    self._load_embedding_model()

    logger.info(f"Initialized RAG System with {embedding_model_name}")
    logger.info(f" Chunk size: {chunk_size}, Overlap is {chunk_overlap}")

  def _load_embedding_model(self):
    #Load embedding model

    try:
      logger.info(f"Loading embedding model: {self.embedding_model_name}")
      self.embedding_model = SentenceTransformer(self.embedding_model_name)
      logger.info(f"Embedding model laoded!")
      except Exception as e:
        logger.error{f"Failed to load embedding model: {str(e)}"}
        raise



In [ ]:
  def load_documents(self, papers_dir: str)-> Dict[str, str]:

    papers_dir = Path(papers_dir)
    if not papers_dir.exist():
      logger error(f"Directory not found: {papers_dir}")
      return {}

    paper_files = [ f for f in os.listdir(papers_dir) if f.endswith('.pdf')]

    if not paper_files:
      logger.error(f"No PDF files found in {papers_dir}")
      return{}


    paper_contents = {}

    for paper_file in paper_files:
      paper_path = os.path.join(papers_dir, paper_file)
      logger.info(f"Loading document: {paper_path}")

      try:

        #Use PyPDFLoader to load pdf doc
        loader = PyPDFLoader(paper_path)
        doc_sections = loader.load()

        if doc_sections:
          content = "\n\n".join([section.page_content for section in doc_sections])
          paper_contents([paper_file]) = content
          logger.info(f"Successfully loaded {paper_file} with ({len(content)} characters)")
        else:
          logger.warning(f"No content loaded from {paaper_file}")

    logger.info(f"Loaded {len(paper_contents)} documents")
    return paper_contents



In [ ]:
  def process_documents(seld, paper_contents: Dict[str, str]):
    logger.info(f"Start Processing Doc....")

    #init text splitter
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap, separators=["\n\n", "\n","","."," "])

    self.documents = []
    self.metadata  = []

    #Process each paper

    for paper_name, content in paper_contents.items():
      logger.info(f"Splitting document: {paper_name}")

      chunks = text_splitter.split_text(content)

      #Store chunks w metadata

      for i, chunk in enumerate(chunks):
        self.documents.append(chunk)
        self.metadatas.append({
            "source": paper_name,
            "chunk_id": i,
            "total_chunks": len(chunks)
        })

      logger.info(f"Created {len(chunks)} chunks from {paper_name}")


    logger.info(f"Processed {len(paper_contents)} documents into {len(self.documents)} chunks")

In [ ]:
def create_embedding(self):
  if not self.documents:
    logger.erro("No documents to create embeddings for")
    return

  logger.info(f"Creating embeddings for {len(self.documents)} chunks...")

  try:
    #generate embeddings for chunks
    #This convert text chunks into numerical vectors
    self.embeddings = self.embeding_model.encode(self.documents, show_progress_bar=True)

    #Convert to numpy for FAISS

    self.embeddings = np.array(self.embeddings).astype('float32')

    logger.info(f"Created embeddings with shape: {elf.embeddings.shape}")
  except Exception as e:
    logger.error(f"Error creating embeddings: {str(e)}")
    raise


def build_faiss_index(self):

  if self.embeddings is None or len(self.embeddings) == 0:
    logger.error("No embeddings to build index with")
    return

  try:
    #Get embedding dimension
    dimension = self.embeddings.shapre[1]

    logger.info(f"Building FAISS index with dimension {dimension}")
    self.index = faiss.IndexFlatL2(dimension)

    self.index.add(self.embeddings)

    logger.info(f"Built FAISS index with {self.index.ntotal} vectors")
  except Exception as e:
    logger.error(f"Error building FAISS index: {str(e)}")
    raise